# Challenge 3 — PPO for ALE/MontezumaRevenge-v5
**Group 1 | Machine Learning — Universidad Distrital**

Este notebook entrena el agente PPO y corre el sweep de hiperparámetros.
Asegúrate de tener activada la GPU: **Entorno de ejecución → T4 GPU**

## 1. Verificar GPU

In [ ]:
import torch
print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('ADVERTENCIA: No hay GPU. Ve a Entorno de ejecución -> Cambiar tipo -> T4 GPU')

## 2. Instalar dependencias

In [ ]:
%%capture
!pip install gymnasium[atari] ale-py opencv-python tensorboard autorom
!AutoROM --accept-license

## 3. Clonar repositorio

In [ ]:
import os

REPO_URL = 'https://github.com/Johan044/challenge1-dqn-atari'
REPO_DIR = '/content/challenge1-dqn-atari'
CHALLENGE_DIR = os.path.join(REPO_DIR, 'challenge3_group1')

if os.path.exists(REPO_DIR):
    print('Repo ya existe, haciendo pull...')
    !cd {REPO_DIR} && git pull
else:
    print('Clonando repo...')
    !git clone {REPO_URL} {REPO_DIR}

print(f'\nArchivos en challenge3_group1:')
!ls {CHALLENGE_DIR}

## 4. Verificar entorno

In [ ]:
import sys
sys.path.insert(0, CHALLENGE_DIR)
os.chdir(CHALLENGE_DIR)

# Sanity check del entorno
!python env_utils.py

## 5. Montar Google Drive (para guardar modelos)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/challenge3_ppo_logs'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive montado. Logs se copiarán a: {DRIVE_DIR}')

## 6. Run único — prueba rápida (50k pasos)

In [ ]:
# Prueba rápida para verificar que todo funciona antes del sweep completo
# Tarda ~2-3 minutos con GPU
!python train.py --total_steps 51200 --seed 42

## 7. Sweep completo (5 configuraciones × 5M pasos)

In [ ]:
# ADVERTENCIA: esto tarda ~5-8 horas en total con GPU T4
# Colab puede desconectarse — si pasa, vuelve a correr esta celda
# Los runs ya completados no se repiten (los checkpoints ya existen)
!python train.py --sweep

## 8. Evaluar el mejor modelo

In [ ]:
import glob

# Listar todos los checkpoints disponibles
checkpoints = glob.glob('logs/montezuma_ppo/**/best_model.pt', recursive=True)
print('Checkpoints disponibles:')
for i, ckpt in enumerate(checkpoints):
    print(f'  [{i}] {ckpt}')

In [ ]:
# Cambiar el índice [0] por el checkpoint que quieras evaluar
BEST_CKPT = checkpoints[0]
print(f'Evaluando: {BEST_CKPT}')

!python evaluate.py --checkpoint "{BEST_CKPT}" --n_episodes 10 --save_results

## 9. Ver curvas de aprendizaje con TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/montezuma_ppo

## 10. Copiar logs y modelos a Google Drive

In [ ]:
import shutil

# Copia todos los logs al Drive para no perderlos si Colab se desconecta
src = os.path.join(CHALLENGE_DIR, 'logs')
dst = os.path.join(DRIVE_DIR, 'logs')

if os.path.exists(dst):
    shutil.rmtree(dst)

shutil.copytree(src, dst)
print(f'Logs copiados a Google Drive: {dst}')
print('\nContenido:')
!ls {dst}

## 11. Graficar curvas de aprendizaje (para el paper)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob

# Cargar returns de todos los runs
returns_files = glob.glob('logs/montezuma_ppo/**/returns.npy', recursive=True)

fig, ax = plt.subplots(figsize=(10, 5))

for f in returns_files:
    run_name = f.split('montezuma_ppo/')[1].split('/returns')[0]
    returns = np.load(f)

    # Rolling mean 100 episodios
    if len(returns) >= 100:
        rolling = np.convolve(returns, np.ones(100)/100, mode='valid')
        ax.plot(rolling, label=run_name[:40], alpha=0.8)
    else:
        ax.plot(returns, label=run_name[:40], alpha=0.8)

ax.set_xlabel('Episode')
ax.set_ylabel('Mean Return (100-ep window)')
ax.set_title('PPO — ALE/MontezumaRevenge-v5')
ax.legend(fontsize=7, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()

# Guardar figura para el paper
fig_path = os.path.join(DRIVE_DIR, 'learning_curves.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Figura guardada en: {fig_path}')
plt.show()